# Whale / Boat / Water Chip Classifier — Evaluation Notebook

Scores the models trained by `train_chip_classifier.ipynb` the way the ICS 635 report guide asks
(`docs/ML-Report-Writing-Guide.pdf`):

| Report standard | Where |
| --- | --- |
| Evaluation: baseline on the same test data, suitable metric, split named for every number, test used once | §3 baselines, §4 thresholds (val only), §5 test results |
| Limitations: where it fails, which groups, label reliability | §6 curves and calibration, §7 error analysis, §8 failures, §9 uncertain whales |
| Ablations: change one thing, same test data | §10 |
| Reproducibility: data, splits, seeds | `run.json` from the training run, printed in §1 |

**Metric.** Classes are imbalanced (water is about half the chips, whales about a tenth), so
plain accuracy mostly reflects water. The headline metric is **macro F1** (each class counts
equally), with **whale recall** and **whale average precision** alongside, since missing whales
is the costly error for a survey. Confidence intervals resample whole test groups (clusters of overlapping
same-day chips), so near-duplicate chips are resampled together.

**Test set discipline.** Everything that is chosen (baseline settings, the whale threshold) is
chosen on the validation split. The test split is predicted once and only reported.

## 1. Setup and run

In [ ]:
import os
import sys

os.environ.setdefault('CUDA_MODULE_LOADING', 'LAZY')
if 'google.colab' in sys.modules:
    !pip install -q rasterio einops timm

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    f1_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()),
    Path.cwd(),
)
sys.path.insert(0, str(ROOT / 'src'))
from whaleboat.data import (  # noqa: E402
    CLASSES,
    load_chips,
    load_skysat,
    overlaps_same_day,
    read_chip,
)
from whaleboat.evaluate import (  # noqa: E402
    WHALE,
    chip_features,
    decide,
    expected_calibration_error,
    group_bootstrap,
    load_run,
    load_trained_model,
    metrics,
    pick_whale_threshold,
    predict_probs,
)

ORIG_DIR = ROOT / 'data' / 'whale' / 'data'
SKYSAT_DIR = ROOT / 'data' / 'skysat_chips_500'
SATVIT_DIR = ROOT / 'satvit'
CKPT_DIR = ROOT / 'checkpoints'

RUN_NAME = 'main'  # the run to evaluate (checkpoints/<RUN_NAME>/)
ABLATION_RUNS = []  # e.g. ['original_only'], trained with SPLIT_FROM = 'main'
NUM_WORKERS = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

RUN_DIR = CKPT_DIR / RUN_NAME
run = load_run(RUN_DIR)
MODEL_NAMES = [
    m for m in ['efficientnet_b0', 'deit_tiny', 'satvit'] if (RUN_DIR / f'{m}.pt').exists()
]

# Rebuild every chip the run used, from the names saved in run.json.
by_name = {c['name']: c for c in load_chips(['original', 'skysat'], ORIG_DIR, SKYSAT_DIR)}
missing = [n for part in run['split'].values() for n in part if n not in by_name]
assert not missing, f'{len(missing)} chips from run.json not found locally, e.g. {missing[:3]}'
sets = {part: [by_name[n] for n in names] for part, names in run['split'].items()}
y = {part: np.array([CLASSES.index(c['label']) for c in cs]) for part, cs in sets.items()}
groups = {part: np.array([c['group'] for c in cs]) for part, cs in sets.items()}

print(f'run {RUN_NAME}: datasets={run["datasets"]} split={run["split_mode"]} seed={run["seed"]}')
print(f'models: {MODEL_NAMES}')
for part, cs in sets.items():
    counts = '  '.join(f'{k}={np.sum(y[part] == i)}' for i, k in enumerate(CLASSES))
    print(f'{part:5s} n={len(cs):4d}   {counts}   groups={len(set(groups[part]))}')

## 2. Model predictions

Softmax probabilities for train, validation and test. Train and validation predictions are used
for the overfitting check and for choosing the threshold; test predictions only for reporting.
They're cached in the run folder.

In [ ]:
probs = {}
for name in MODEL_NAMES:
    cache = RUN_DIR / f'{name}_probs.npz'
    if cache.exists():
        probs[name] = dict(np.load(cache))
        continue
    model, ckpt = load_trained_model(RUN_DIR, name, SATVIT_DIR, DEVICE)
    probs[name] = {
        part: predict_probs(model, name, cs, run['band_lo'], run['band_hi'], DEVICE, NUM_WORKERS)
        for part, cs in sets.items()
    }
    np.savez(cache, **probs[name])
    del model
    torch.cuda.empty_cache()
    print(f'{name}: best epoch {ckpt["epoch"]}')

## 3. Baselines

Both are fit on the training split; anything chosen is chosen on validation.

- **Majority class:** always predict the most common training class. Shows what a useless model
  scores on each metric.
- **Logistic regression on chip statistics:** 15 hand-made features per chip (per-band log mean,
  std and 99th percentile; the strongest local brightness anomaly and the number of anomaly
  blobs; valid-pixel share; see `whaleboat.evaluate.chip_features`). Standardized, class-weighted,
  `C` picked from {0.01, 0.1, 1, 10, 100} by validation macro F1. It asks whether a deep model
  beats "is there a bright blob in the water".

In [ ]:
feats = {part: np.stack([chip_features(c['path']) for c in cs]) for part, cs in sets.items()}

majority = np.bincount(y['train'], minlength=len(CLASSES)).argmax()
baseline_probs = {'majority': {p: np.eye(len(CLASSES))[np.full(len(y[p]), majority)] for p in sets}}

best_c, best_f1 = None, -1
for c in [0.01, 0.1, 1, 10, 100]:
    lr = make_pipeline(
        StandardScaler(), LogisticRegression(C=c, class_weight='balanced', max_iter=5000)
    )
    lr.fit(feats['train'], y['train'])
    f1 = metrics(y['val'], lr.predict(feats['val']))['macro_f1']
    print(f'  C={c:<6} val macro F1={f1:.3f}')
    if f1 > best_f1:
        best_c, best_f1, best_lr = c, f1, lr
print(f'logistic regression: C={best_c} (val macro F1 {best_f1:.3f})')
baseline_probs['logreg'] = {p: best_lr.predict_proba(feats[p]) for p in sets}
all_probs = {**baseline_probs, **probs}

## 4. Decision threshold (validation only)

Whales are rare and missing one is costly, so besides plain argmax each model gets a whale
threshold: predict whale whenever p(whale) ≥ t. `t` is the value that maximizes **validation**
macro F1, and is then fixed for the test set.

In [ ]:
thresholds = {}
for name, pr in all_probs.items():
    if name == 'majority':
        thresholds[name] = None
        continue
    t, f1 = pick_whale_threshold(pr['val'], y['val'])
    thresholds[name] = t
    argmax_f1 = metrics(y['val'], decide(pr['val']))['macro_f1']
    print(f'{name:16s} whale threshold={t:.2f}  val macro F1: argmax={argmax_f1:.3f} → {f1:.3f}')

## 5. Test results

Test split, scored once with the settings fixed above. `n` is the number of test chips;
intervals are 95% group-bootstrap CIs over the test groups. Train and validation macro F1 are
shown for the overfitting check (same metric, same decision rule).

In [ ]:
rows = []
for name, pr in all_probs.items():
    t = thresholds[name]
    pred = {p: decide(pr[p], t) for p in sets}
    m = metrics(y['test'], pred['test'], pr['test'])
    ci = group_bootstrap(y['test'], pred['test'], groups['test'], pr['test'])
    m['train_macro_f1'] = metrics(y['train'], pred['train'])['macro_f1']
    m['val_macro_f1'] = metrics(y['val'], pred['val'])['macro_f1']
    m['ece'] = expected_calibration_error(pr['test'], y['test'])
    rows.append((name, m, ci))

print(f'TEST split: n={len(y["test"])} chips, {len(set(groups["test"]))} groups')
hdr = (
    f'{"model":16s} {"macro F1 [95% CI]":>22s} {"whale rec [95% CI]":>22s} '
    f'{"whale AP":>9s} {"acc":>6s} {"ECE":>6s} {"train F1":>9s} {"val F1":>7s}'
)
print(hdr)
for name, m, ci in rows:
    f1ci = f'{m["macro_f1"]:.3f} [{ci["macro_f1"][0]:.2f}, {ci["macro_f1"][1]:.2f}]'
    wrci = f'{m["whale_recall"]:.3f} [{ci["whale_recall"][0]:.2f}, {ci["whale_recall"][1]:.2f}]'
    print(
        f'{name:16s} {f1ci:>22s} {wrci:>22s} {m.get("whale_ap", float("nan")):9.3f} '
        f'{m["accuracy"]:6.3f} {m["ece"]:6.3f} {m["train_macro_f1"]:9.3f} {m["val_macro_f1"]:7.3f}'
    )

print('\nper-class precision / recall / F1 (test):')
for name, m, _ in rows:
    cls = '  '.join(
        f'{c}={m[f"{c}_precision"]:.2f}/{m[f"{c}_recall"]:.2f}/{m[f"{c}_f1"]:.2f}' for c in CLASSES
    )
    print(f'  {name:16s} {cls}')

In [ ]:
fig, axes = plt.subplots(1, len(all_probs), figsize=(4 * len(all_probs), 4))
for ax, (name, pr) in zip(np.atleast_1d(axes), all_probs.items()):
    ConfusionMatrixDisplay.from_predictions(
        y['test'],
        decide(pr['test'], thresholds[name]),
        labels=range(len(CLASSES)),
        display_labels=CLASSES,
        ax=ax,
        colorbar=False,
    )
    ax.set_title(f'{name} (test, n={len(y["test"])})', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Curves and calibration (test)

One-vs-rest precision-recall and ROC curves per class. PR is the more informative of the two for
the rare whale class. The reliability diagram compares the top-class confidence with how often
that prediction is right; ECE (in the table above) summarizes the gap.

In [ ]:
curve_models = [n for n in all_probs if n != 'majority']
fig, axes = plt.subplots(2, len(CLASSES), figsize=(5 * len(CLASSES), 9))
for k, c in enumerate(CLASSES):
    for name in curve_models:
        PrecisionRecallDisplay.from_predictions(
            y['test'] == k, all_probs[name]['test'][:, k], name=name, ax=axes[0, k]
        )
        RocCurveDisplay.from_predictions(
            y['test'] == k, all_probs[name]['test'][:, k], name=name, ax=axes[1, k]
        )
    axes[0, k].set_title(f'{c}: precision-recall (test, n={len(y["test"])})')
    axes[1, k].set_title(f'{c}: ROC (test)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], 'k:', label='perfect')
for name in curve_models:
    pr = all_probs[name]['test']
    frac, conf = calibration_curve(
        pr.argmax(1) == y['test'], pr.max(1), n_bins=10, strategy='quantile'
    )
    plt.plot(conf, frac, 'o-', label=name)
plt.xlabel('top-class confidence')
plt.ylabel('fraction correct')
plt.title(f'Reliability (test, n={len(y["test"])})')
plt.legend()
plt.show()

## 7. Error analysis by group and condition (test)

Where does each model do worse? Macro F1 and whale recall per data source and per acquisition date, and
recall per class across imaging conditions from `manifest_conditions.csv` (SkySat chips only):
heavy haze (UDM2), water roughness (NIR spread, terciles), and sun elevation. For whales, recall
by annotator certainty (1 = likely, 2 = clearly a whale). Macro F1 here averages only the
classes present in that subgroup. Small groups give noisy numbers; `n` is
shown for each.

In [ ]:
conditions = {}
cond_path = SKYSAT_DIR / 'main' / 'manifest_conditions.csv'
if cond_path.exists():
    with open(cond_path, newline='') as f:
        conditions = {f'skysat/{r["chip_id"]}': r for r in csv.DictReader(f)}
test_names = [c['name'] for c in sets['test']]


def cond_value(name, key):
    v = conditions.get(name, {}).get(key, '')
    try:
        return float(v)
    except ValueError:
        return np.nan


def breakdown(name, keys, title):
    pred = decide(all_probs[name]['test'], thresholds[name])
    keys = np.array(keys, dtype=object)
    print(f'\n{name} — {title}')
    for k in sorted(set(keys), key=str):
        m = keys == k
        if m.sum() == 0:
            continue
        r = metrics(y['test'][m], pred[m])
        present = sorted(set(y['test'][m]))
        f1 = f1_score(y['test'][m], pred[m], labels=present, average='macro', zero_division=0)
        recalls = '  '.join(
            f'{c}_rec={r[f"{c}_recall"]:.2f}(n={np.sum(y["test"][m] == i)})'
            for i, c in enumerate(CLASSES)
            if np.sum(y['test'][m] == i)
        )
        print(f'  {str(k):28s} n={m.sum():4d}  macro F1={f1:.3f}  {recalls}')


rough = np.array([cond_value(n, 'nir_std') for n in test_names])
cuts = np.nanpercentile(rough, [33, 67]) if np.isfinite(rough).any() else [np.nan, np.nan]
for name in curve_models:
    breakdown(name, [c['source'] for c in sets['test']], 'by data source')
    breakdown(name, [c['date'] for c in sets['test']], 'by acquisition date')
    haze = [cond_value(n, 'heavy_haze_frac') for n in test_names]
    breakdown(
        name,
        ['n/a' if h != h else ('heavy haze' if h > 0.5 else 'little haze') for h in haze],
        'by haze',
    )
    breakdown(
        name,
        [
            'n/a' if r != r else ('calm' if r < cuts[0] else 'medium' if r < cuts[1] else 'rough')
            for r in rough
        ],
        'by water roughness (NIR std terciles)',
    )
    sun = [cond_value(n, 'sun_elevation') for n in test_names]
    breakdown(
        name,
        ['n/a' if s != s else ('sun < 50°' if s < 50 else 'sun ≥ 50°') for s in sun],
        'by sun elevation',
    )
    cert = [conditions.get(n, {}).get('whale_certainty_max', '') or 'n/a' for n in test_names]
    breakdown(
        name,
        [f'whale certainty {c}' if c != 'n/a' else 'not whale / original' for c in cert],
        'by whale certainty',
    )

## 8. Failure examples (test)

The most confident wrong predictions of each model, as true-color crops. These are the cases
to describe in the limitations section (and to check against the labels: some "errors" may be
label mistakes).

In [ ]:
def show_rgb(ax, path):
    a = read_chip(path)[[2, 1, 0]].astype(np.float32)
    lo, hi = np.percentile(a[a > 0], [2, 98]) if (a > 0).any() else (0, 1)
    ax.imshow(np.clip((a.transpose(1, 2, 0) - lo) / (hi - lo + 1e-6), 0, 1))
    ax.axis('off')


N_SHOW = 8
for name in curve_models:
    pr = all_probs[name]['test']
    pred = decide(pr, thresholds[name])
    wrong = np.flatnonzero(pred != y['test'])
    wrong = wrong[np.argsort(-pr[wrong, pred[wrong]])][:N_SHOW]
    if len(wrong) == 0:
        continue
    fig, axes = plt.subplots(1, len(wrong), figsize=(2.6 * len(wrong), 3), squeeze=False)
    for ax, i in zip(axes[0], wrong):
        show_rgb(ax, sets['test'][i]['path'])
        ax.set_title(
            f'true {CLASSES[y["test"][i]]}\npred {CLASSES[pred[i]]} ({pr[i, pred[i]]:.2f})',
            fontsize=8,
        )
    fig.suptitle(f'{name}: most confident test errors')
    plt.tight_layout()
    plt.show()

## 9. Uncertain whales (never trained on)

The 251 chips in `skysat_chips_500/uncertain_whales/` hold only whales the annotator marked
"possibly a whale". There is no ground truth for them, so this is a reading, not a score: how
often each model calls them whale, next to the same rate for certain whales in the test split.
Uncertain chips that overlap a same-day training chip are excluded, so the model hasn't seen
that exact patch of water.

In [ ]:
uncertain = [
    c
    for c in load_skysat(SKYSAT_DIR, 'uncertain_whales')
    if not overlaps_same_day(c, sets['train'])
]
print(f'{len(uncertain)} uncertain-whale chips not overlapping a training chip')
test_whales = y['test'] == WHALE
for name in MODEL_NAMES:
    model, _ = load_trained_model(RUN_DIR, name, SATVIT_DIR, DEVICE)
    pu = predict_probs(model, name, uncertain, run['band_lo'], run['band_hi'], DEVICE, NUM_WORKERS)
    del model
    torch.cuda.empty_cache()
    t = thresholds[name]
    unc_rate = np.mean(decide(pu, t) == WHALE) if len(pu) else np.nan
    cert_rate = np.mean(decide(all_probs[name]['test'][test_whales], t) == WHALE)
    print(
        f'{name:16s} called whale: uncertain {unc_rate:.2f} (n={len(pu)})   '
        f'certain test whales {cert_rate:.2f} (n={test_whales.sum()})'
    )

## 10. Learning curves and ablations

Learning curves come from the training history saved in each checkpoint (validation split).

For ablations, train another run with `SPLIT_FROM = 'main'` and one change (for example
`DATASETS = ['original']`, or `SPLIT_MODE = 'random'` to show what leakage does), then list it
in `ABLATION_RUNS`. Every run is scored on **this run's test chips**, so differences come from
the change, not the test data. Runs whose training chips overlap this test set are flagged.

In [ ]:
for name in MODEL_NAMES:
    hist = torch.load(RUN_DIR / f'{name}.pt', map_location='cpu', weights_only=False)['history']
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].plot(hist['train_loss'], label='train')
    axes[0].plot(hist['val_loss'], label='validation')
    axes[0].set_ylabel('weighted cross-entropy')
    axes[1].plot(hist['train_f1'], label='train')
    axes[1].plot(hist['val_f1'], label='validation')
    axes[1].set_ylabel('macro F1 (argmax)')
    for ax in axes:
        ax.set_xlabel('epoch')
        ax.legend()
    fig.suptitle(f'{name} learning curves ({RUN_NAME})')
    plt.tight_layout()
    plt.show()

In [ ]:
test_set = set(run['split']['test'])
print(f'Ablations scored on the TEST split of run {RUN_NAME} (n={len(y["test"])}), argmax decision')
print(f'{"run":20s} {"model":16s} {"datasets":>22s} {"macro F1":>9s} {"whale rec":>10s}')
for run_name in [RUN_NAME, *ABLATION_RUNS]:
    rdir = CKPT_DIR / run_name
    r = load_run(rdir)
    leak = len(test_set & set(r['split']['train']))
    for name in [
        m for m in ['efficientnet_b0', 'deit_tiny', 'satvit'] if (rdir / f'{m}.pt').exists()
    ]:
        model, _ = load_trained_model(rdir, name, SATVIT_DIR, DEVICE)
        pr = predict_probs(
            model, name, sets['test'], r['band_lo'], r['band_hi'], DEVICE, NUM_WORKERS
        )
        del model
        torch.cuda.empty_cache()
        m = metrics(y['test'], decide(pr), pr)
        flag = f'  ⚠ {leak} test chips were in its training split' if leak else ''
        ds = '+'.join(r['datasets'])
        print(
            f'{run_name:20s} {name:16s} {ds:>22s} {m["macro_f1"]:9.3f} '
            f'{m["whale_recall"]:10.3f}{flag}'
        )